In [5]:
from dotenv import load_dotenv
import os
import re
from llama_cloud import CohereEmbedding
from pinecone import Pinecone
from llama_index.core import (
    VectorStoreIndex, 
    StorageContext, 
    Settings, 
    PromptTemplate, 
    get_response_synthesizer
)
from llama_index.vector_stores.pinecone import PineconeVectorStore
from llama_index.llms.gemini import Gemini
from llama_index.core.workflow import Workflow, step, StartEvent, StopEvent, Event
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.embeddings.cohere import CohereEmbedding

load_dotenv()

COHERE_API_KEY = os.getenv("COHERE_API_KEY")
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")


Settings.llm = Gemini(
    api_key=GOOGLE_API_KEY, 
    model_name="models/gemini-2.5-flash", 
    temperature=0.1 ,
    transport="rest"
)

embed_model = CohereEmbedding(
    api_key=COHERE_API_KEY,
    model_name="embed-multilingual-v3.0",
)

Settings.embed_model = embed_model

pc = Pinecone(
    api_key=PINECONE_API_KEY,
    ssl_verify=False  
)
pinecone_index = pc.Index("rag-lesson")
vector_store = PineconeVectorStore(pinecone_index=pinecone_index,namespace="")
storage_context = StorageContext.from_defaults(vector_store=vector_store)

index = VectorStoreIndex.from_vector_store(
    vector_store, 
    storage_context=storage_context
)

qa_prompt_tmpl_str = (
    "Context information is below.\n"
    "---------------------\n"
    "{context_str}\n"
    "---------------------\n"
    "You are a professional technical assistant. Based ONLY on the context provided above (and not your general knowledge), "
    "answer the following query. \n"
    "CRITICAL RULES:\n"
    "1. If the answer is not in the context, say: 'מצטער, המידע הזה לא מופיע בתיעוד הפרויקט'.\n"
    "2. Always respond in the SAME LANGUAGE as the user's question.\n"
    "3. If there are contradictions between tools (e.g., Windsurf vs Copilot), highlight them.\n"
    "Query: {query_str}\n"
    "Answer: "
)
qa_prompt_tmpl = PromptTemplate(qa_prompt_tmpl_str)

response_synthesizer = get_response_synthesizer(
    response_mode="tree_summarize",
    text_qa_template=qa_prompt_tmpl
)

print("✅ תשתית שלב ב' מוכנה לחלוטין! כולל מודל ה-Embedding של Cohere.")

C:\Users\user1\AppData\Local\Temp\ipykernel_30788\4117291284.py:26: DeprecationWarning: Call to deprecated class Gemini. (Should use `llama-index-llms-google-genai` instead, using Google's latest unified SDK. See: https://docs.llamaindex.ai/en/stable/examples/llm/google_genai/This package will no longer be supported after version 0.6.2) -- Deprecated since version 0.6.2.
  Settings.llm = Gemini(


✅ תשתית שלב ב' מוכנה לחלוטין! כולל מודל ה-Embedding של Cohere.


In [6]:
from llama_index.core.workflow import Event
from llama_index.core.schema import NodeWithScore

class InputVerifiedEvent(Event):
    query: str

class ContextRetrievedEvent(Event):
    query: str
    nodes: list[NodeWithScore]

class LowConfidenceEvent(Event):
    query: str

In [7]:
class RAGWorkflow(Workflow):

    @step
    async def security_and_validation(self, ev: StartEvent) -> InputVerifiedEvent | StopEvent:
        query = ev.query.strip()
        
        if len(query) < 2 or len(query) > 500:
            return StopEvent(result="שגיאה: קלט באורך לא תקין.")
        
        clean_query = re.sub(r'<[^>]*?>', '', query)

        malicious_patterns = ["ignore previous instructions", "system prompt", "as an admin", "bypass"]
        if any(pattern in query.lower() for pattern in malicious_patterns):
            return StopEvent(result="מצטער, השאילתה שלך מכילה פקודות לא מורשות.")

        clean_query = "".join(char for char in query if char.isalnum() or char in " ,.?!")
        
        return InputVerifiedEvent(query=clean_query)

    @step
    async def retrieve_with_retry(self, ev: InputVerifiedEvent) -> ContextRetrievedEvent | LowConfidenceEvent:
        retriever = VectorIndexRetriever(index=index, similarity_top_k=3)
        nodes = await retriever.aretrieve(ev.query)
        
        if not nodes or nodes[0].score < 0.5:
            return LowConfidenceEvent(query=ev.query)
        
        return ContextRetrievedEvent(query=ev.query, nodes=nodes)

    @step
    async def handle_low_confidence(self, ev: LowConfidenceEvent) -> InputVerifiedEvent:
        prompt = f"Rewrite this technical query to be more specific for a vector search: {ev.query}"

        result = await Settings.llm.acomplete(prompt)
        rewritten_query = str(result)
        return InputVerifiedEvent(query=rewritten_query)

    @step
    async def generate_and_verify_output(self, ev: ContextRetrievedEvent) -> StopEvent:
        response_obj = await response_synthesizer.asynthesize(query=ev.query, nodes=ev.nodes)
        response_str = str(response_obj)

        verify_prompt = f"Is the following answer based ONLY on the context? Answer 'YES' or 'NO'.\nAnswer: {response_str}"
        verification = await Settings.llm.acomplete(verify_prompt)
        
        if "NO" in str(verification).upper():
            return StopEvent(result="מצטער, לא נמצא מידע מהימן מספיק במסמכי הפרויקט.")
            
        print("✨ התשובה עברה בקרת איכות סופית.")
        return StopEvent(result=str(response))

rag_engine = RAGWorkflow(timeout=60)
print("✅ תא 3: ה-Workflow המאובטח מוכן.")

✅ תא 3: ה-Workflow המאובטח מוכן.


In [ ]:
import gradio as gr

# עיצוב Dark-Tech יוקרתי ומקצועי
tech_css = """
/* רקע כללי כהה והייטקי */
.gradio-container { 
    direction: rtl; 
    background-color: #0b0f19 !important; 
    font-family: 'Inter', 'Segoe UI', sans-serif !important;
}

/* כותרת מעוצבת */
#header-text { 
    text-align: center; 
    color: #e2e8f0; 
    padding: 20px;
    border-bottom: 1px solid #1e293b;
}

/* עיצוב בועות הצ'אט */
.message { 
    border-radius: 8px !important; 
    font-size: 0.95rem !important;
    line-height: 1.6 !important;
}

/* בועת המשתמש - כחול הייטק עדין */
.user { 
    background-color: #1e293b !important; 
    color: #f1f5f9 !important; 
    border: 1px solid #334155 !important;
}

/* בועת הבוט - אפור כהה מקצועי */
.bot { 
    background-color: #0f172a !important; 
    color: #cbd5e1 !important; 
    border: 1px solid #1e293b !important;
}

/* עיצוב שדה הטקסט */
#query-input textarea {
    background-color: #1e293b !important;
    color: #f1f5f9 !important;
    border: 1px solid #334155 !important;
    border-radius: 4px !important;
}

/* כפתורי שליטה - מראה שטוח ונקי */
.lg.primary {
    background: linear-gradient(90deg, #3b82f6 0%, #2563eb 100%) !important;
    border: none !important;
}

footer { display: none !important; }
"""

async def chat_response(message, history):
    try:
        response = await rag_engine.run(query=message)
        return str(response)
    except Exception as e:
        return f"Error: {str(e)}"
    
    

# בניית הממשק במבנה Blocks
# בניית הממשק במבנה Blocks הכי מעודכן ל-2026
with gr.Blocks(css=tech_css, theme=gr.themes.Default()) as demo:
    with gr.Row(elem_id="header-text"):
        gr.Markdown("""
        # 💠 PROJECT KNOWLEDGE GRAPH
        **Advanced RAG Interface | Gemini 2.5 Pro**
        """)
    
    with gr.Column():
        # יצירת הצ'אטבוט
        chatbot = gr.Chatbot(
            show_label=False, 
            height=550
        )
        
        # יצירת ממשק הצ'אט עם השמות המעודכנים של הכפתורים
        chat_interface = gr.ChatInterface(
            fn=chat_response,
            chatbot=chatbot,
            textbox=gr.Textbox(
                placeholder="הזן שאילתה טכנית לניתוח הנתונים...", 
                elem_id="query-input",
                container=False, 
                scale=7
            ),
            # בגרסה 5 משתמשים בשמות האלו:
            submit_btn="run query",
            stop_btn="stop",
        )

# הרצה
if __name__ == "__main__":
    demo.launch(debug=True)

C:\Users\user1\AppData\Local\Temp\ipykernel_30788\1668715275.py:69: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=tech_css, theme=gr.themes.Default()) as demo:
C:\Users\user1\AppData\Local\Temp\ipykernel_30788\1668715275.py:84: UserWarning: You provided a custom `textbox` component, but also specified `submit_btn`, `stop_btn` parameter(s) on `gr.ChatInterface`. These ChatInterface parameters will be ignored. To customize these settings, pass them directly to your `gr.Textbox` or `gr.MultimodalTextbox` component instead. For example: textbox=gr.Textbox(..., submit_btn='submit')
  chat_interface = gr.ChatInterface(


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


c:\Users\user1\Desktop\תכנותתת\שנה ב\AI\RAG\rag-project\venv\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'rag-lesson-t3b067q.svc.aped-4627-b74a.pinecone.io'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
